In [75]:
import pandas as pd

idx = 89
vocal_audio_dir = f'../data/clean/audio/vocals/{idx}.mp3'
raw_audio_dir = f'../data/raw/audio/{idx}.mp3'

segment_index_file = f'../data/clean/syllables/segment_index.csv'
segment_index = pd.read_csv(segment_index_file)
segment_index = segment_index[segment_index['file'] == idx]
segment_index.head()

,index,file,start,end,token,pred
29234,29234,89,0,1552,<silence>,<silence>
29235,29235,89,1552,1719,ゆ,い
29236,29236,89,1719,1885,め,な
29237,29237,89,1885,2186,な,な
29238,29238,89,2186,2386,ら,い


In [76]:
def hiragana_to_romaji(token):
    token_set =  {
        # Basic vowels
        'あ': 'a',    'い': 'i',    'う': 'u',    'え': 'e',    'お': 'o',

        # Basic consonant + vowel
        'か': 'ka',   'き': 'ki',   'く': 'ku',   'け': 'ke',   'こ': 'ko',
        'さ': 'sa',   'し': 'shi',  'す': 'su',   'せ': 'se',   'そ': 'so',
        'た': 'ta',   'ち': 'chi',  'つ': 'tsu',  'て': 'te',   'と': 'to',
        'な': 'na',   'に': 'ni',   'ぬ': 'nu',   'ね': 'ne',   'の': 'no',
        'は': 'ha',   'ひ': 'hi',   'ふ': 'fu',   'へ': 'he',   'ほ': 'ho',
        'ま': 'ma',   'み': 'mi',   'む': 'mu',   'め': 'me',   'も': 'mo',
        'や': 'ya',   'ゆ': 'yu',   'よ': 'yo',
        'ら': 'ra',   'り': 'ri',   'る': 'ru',   'れ': 're',   'ろ': 'ro',
        'わ': 'wa',   'を': 'wo',   'ん': 'n',

        # Dakuten (voiced) sounds
        'が': 'ga',   'ぎ': 'gi',   'ぐ': 'gu',   'げ': 'ge',   'ご': 'go',
        'ざ': 'za',   'じ': 'ji',   'ず': 'zu',   'ぜ': 'ze',   'ぞ': 'zo',
        'だ': 'da',   'ぢ': 'ji',   'づ': 'zu',   'で': 'de',   'ど': 'do',
        'ば': 'ba',   'び': 'bi',   'ぶ': 'bu',   'べ': 'be',   'ぼ': 'bo',

        # Handakuten (semi-voiced) sounds
        'ぱ': 'pa',   'ぴ': 'pi',   'ぷ': 'pu',   'ぺ': 'pe',   'ぽ': 'po',

        # Yōon (combined sounds)
        'きゃ': 'kya', 'きゅ': 'kyu', 'きょ': 'kyo',
        'しゃ': 'sha', 'しゅ': 'shu', 'しょ': 'sho',
        'ちゃ': 'cha', 'ちゅ': 'chu', 'ちょ': 'cho',
        'にゃ': 'nya', 'にゅ': 'nyu', 'にょ': 'nyo',
        'ひゃ': 'hya', 'ひゅ': 'hyu', 'ひょ': 'hyo',
        'みゃ': 'mya', 'みゅ': 'myu', 'みょ': 'myo',
        'りゃ': 'rya', 'りゅ': 'ryu', 'りょ': 'ryo',
        'ぎゃ': 'gya', 'ぎゅ': 'gyu', 'ぎょ': 'gyo',
        'じゃ': 'ja',  'じゅ': 'ju',  'じょ': 'jo',
        'びゃ': 'bya', 'びゅ': 'byu', 'びょ': 'byo',
        'ぴゃ': 'pya', 'ぴゅ': 'pyu', 'ぴょ': 'pyo',

        # Additional special sounds
        'でぃ': 'di',
        'ふぁ': 'fa',  'ふぃ': 'fi',  'ふぇ': 'fe',  'ふぉ': 'fo',

        # Special tokens
        '<silence>': '[silence]',
        '<bre>': '[breath]',
        '<echo>': '[echo]'
    }
    
    return token_set.get(token, token)


In [77]:
segment_index['token'] = segment_index['token'].map(hiragana_to_romaji)
segment_index['pred'] = segment_index['pred'].map(hiragana_to_romaji)
segment_index.head()

,index,file,start,end,token,pred
29234,29234,89,0,1552,[silence],[silence]
29235,29235,89,1552,1719,yu,i
29236,29236,89,1719,1885,me,na
29237,29237,89,1885,2186,na,na
29238,29238,89,2186,2386,ra,i


In [78]:
# def ms_to_srt_time(ms):
#     """Convert milliseconds to SRT timestamp format"""
#     seconds = ms // 1000
#     ms = ms % 1000
#     minutes = seconds // 60
#     seconds = seconds % 60
#     hours = minutes // 60
#     minutes = minutes % 60
#     return f"{hours:02d}:{minutes:02d}:{seconds:02d},{ms:03d}"
# 
# def create_srt(segment_index):
#     srt_entries = []
# 
#     for idx, row in segment_index.iterrows():
#         counter = idx - segment_index.index[0] + 1
#         start_time = ms_to_srt_time(row['start'])
#         end_time = ms_to_srt_time(row['end'])
# 
#         # Color coding: green for token, white/red for pred based on match
#         token = f'\033[32m]{row["token"]}\033[0m]'  # green
#         if row['token'] == row['pred']:
#             pred = f'{row["pred"]}'  # white
#         else:
#             pred = f'\033[31m]{row["pred"]}\033[0m]'  # red
# 
#         text = f"{token}\n{pred}"
# 
#         srt_entry = f"{counter}\n{start_time} --> {end_time}\n{text}\n"
#         srt_entries.append(srt_entry)
# 
#     return "\n".join(srt_entries)
# 
# # Generate SRT content
# srt_content = create_srt(segment_index)
# 
# # Write to file
# output_file = f'{idx}.srt'
# with open(output_file, 'w', encoding='utf-8') as f:
#     f.write(srt_content)


In [79]:
def ms_to_vtt_time(ms):
    """Convert milliseconds to WebVTT timestamp format"""
    seconds = ms / 1000
    minutes = int(seconds // 60)
    seconds = seconds % 60
    hours = int(minutes // 60)
    minutes = minutes % 60
    return f"{hours:02d}:{minutes:02d}:{seconds:06.3f}"

def create_vtt(segment_index):
    # WebVTT header with styling
    vtt_header = """WEBVTT

STYLE
::cue {
    font-size: 200%;
    line-height: 1.5;
    background-color: transparent;
}
::cue(.green) {
    color: lime;
}
::cue(.red) {
    color: red;
}

"""
    vtt_entries = [vtt_header]

    for idx, row in segment_index.iterrows():
        start_time = ms_to_vtt_time(row['start'])
        end_time = ms_to_vtt_time(row['end'])

        token = f'<c.green>{row["token"]}</c>'
        if row['token'] == row['pred']:
            pred = f'<c.green>{row["pred"]}</c>'  # default color (white)
        else:
            pred = f'<c.red>{row["pred"]}</c>'

        text = f"{token}\n{pred}"

        vtt_entry = f"{start_time} --> {end_time}\n{text}\n\n"
        vtt_entries.append(vtt_entry)

    return "".join(vtt_entries)

# Generate VTT content
vtt_content = create_vtt(segment_index)

# Write to file
for i in [4, 89]:
    output_file = f'{idx}.vtt'
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(vtt_content)


In [73]:
def create_line_by_line_vtt(segment_index, max_line_length=10, line_duration=1000):
    """
    Create a WebVTT file with colored line-by-line subtitles
    """
    def ms_to_vtt_time(ms):
        """Convert milliseconds to WebVTT timestamp format"""
        seconds = ms / 1000
        minutes = int(seconds // 60)
        seconds = seconds % 60
        hours = int(minutes // 60)
        minutes = minutes % 60
        return f"{hours:02d}:{minutes:02d}:{seconds:06.3f}"

    # WebVTT header with advanced styling
    vtt_header = """WEBVTT

STYLE
::cue {
    background-color: transparent;
    color: white;
    font-size: 1.5em;
    font-family: Arial, sans-serif;
}
::cue(.correct) {
    color: lime;
}
::cue(.incorrect) {
    color: red;
}
::cue(.predicted) {
    color: cyan;
}

"""
    vtt_entries = [vtt_header]

    current_line = []

    for _, row in segment_index.iterrows():
        # Add to current line
        current_line.append({
            'token': row['token'],
            'pred': row['pred'],
            'start': row['start'],
            'end': row['end']
        })

        # Check if line is ready to be written
        if len(''.join([item['token'] for item in current_line])) >= max_line_length:
            # Write the line
            line_start = current_line[0]['start']
            line_end = current_line[-1]['end'] + line_duration

            # Construct colored line text
            line_text_parts = []
            line_pred_parts = []

            for item in current_line:
                # Determine color class for tokens
                token_class = 'correct' if item['token'] == item['pred'] else 'incorrect'

                # Add colored tokens
                line_text_parts.append(f'<c.{token_class}>{item["token"]}</c>')
                line_pred_parts.append(f'<c.predicted>{item["pred"]}</c>')

            # Create VTT entry
            vtt_entry = (
                f"{ms_to_vtt_time(line_start)} --> {ms_to_vtt_time(line_end)}\n"
                f"{' '.join(line_text_parts)}\n"
                f"{' '.join(line_pred_parts)}\n\n"
            )
            vtt_entries.append(vtt_entry)

            # Reset current line
            current_line = []

    # Handle any remaining tokens
    if current_line:
        line_start = current_line[0]['start']
        line_end = current_line[-1]['end'] + line_duration

        line_text_parts = []
        line_pred_parts = []

        for item in current_line:
            # Determine color class for tokens
            token_class = 'correct' if item['token'] == item['pred'] else 'incorrect'

            # Add colored tokens
            line_text_parts.append(f'<c.{token_class}>{item["token"]}</c>')
            line_pred_parts.append(f'<c.predicted>{item["pred"]}</c>')

        vtt_entry = (
            f"{ms_to_vtt_time(line_start)} --> {ms_to_vtt_time(line_end)}\n"
            f"{' '.join(line_text_parts)}\n"
            f"{' '.join(line_pred_parts)}\n\n"
        )
        vtt_entries.append(vtt_entry)

    return "".join(vtt_entries)

# Usage remains the same
def generate_subtitles(segment_index, output_path):
    vtt_content = create_line_by_line_vtt(
        segment_index,
        max_line_length=15,  # Adjust based on your preference
        line_duration=1500   # How long each line stays (ms)
    )

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(vtt_content)


In [74]:
for i in [4, 89]:
    output_file = f'{i}.vtt'
    generate_subtitles(segment_index, output_file)

In [71]:
def create_line_by_line_vtt(segment_index, max_line_length=10, line_duration=1000):
    """
    Create a WebVTT file with colored line-by-line subtitles
    """
    def ms_to_vtt_time(ms):
        """Convert milliseconds to WebVTT timestamp format"""
        seconds = ms / 1000
        minutes = int(seconds // 60)
        seconds = seconds % 60
        hours = int(minutes // 60)
        minutes = minutes % 60
        return f"{hours:02d}:{minutes:02d}:{seconds:06.3f}"

    # WebVTT header with advanced styling
    vtt_header = """WEBVTT

STYLE
::cue {
    background-color: transparent;
    color: white;
    font-size: 1.5em;
    font-family: Arial, sans-serif;
}
::cue(.correct) {
    color: lime;
}
::cue(.incorrect) {
    color: red;
}
::cue(.predicted) {
    color: cyan;
}

"""
    vtt_entries = [vtt_header]

    current_line = []

    for _, row in segment_index.iterrows():
        # Add to current line
        current_line.append({
            'token': row['token'],
            'pred': row['pred'],
            'start': row['start'],
            'end': row['end']
        })

        # Check if line is ready to be written
        if len(''.join([item['token'] for item in current_line])) >= max_line_length:
            # Write the line
            line_start = current_line[0]['start']
            line_end = current_line[-1]['end'] + line_duration

            # Construct colored line text
            line_text_parts = []
            line_pred_parts = []

            for item in current_line:
                # Determine color class for tokens
                token_class = 'correct' if item['token'] == item['pred'] else 'incorrect'

                # Add colored tokens
                line_text_parts.append(f'<c.{token_class}>{item["token"]}</c>')
                line_pred_parts.append(f'<c.predicted>{item["pred"]}</c>')

            # Create VTT entry
            vtt_entry = (
                f"{ms_to_vtt_time(line_start)} --> {ms_to_vtt_time(line_end)}\n"
                f"{' '.join(line_text_parts)}\n"
                f"{' '.join(line_pred_parts)}\n\n"
            )
            vtt_entries.append(vtt_entry)

            # Reset current line
            current_line = []

    # Handle any remaining tokens
    if current_line:
        line_start = current_line[0]['start']
        line_end = current_line[-1]['end'] + line_duration

        line_text_parts = []
        line_pred_parts = []

        for item in current_line:
            # Determine color class for tokens
            token_class = 'correct' if item['token'] == item['pred'] else 'incorrect'

            # Add colored tokens
            line_text_parts.append(f'<c.{token_class}>{item["token"]}</c>')
            line_pred_parts.append(f'<c.predicted>{item["pred"]}</c>')

        vtt_entry = (
            f"{ms_to_vtt_time(line_start)} --> {ms_to_vtt_time(line_end)}\n"
            f"{' '.join(line_text_parts)}\n"
            f"{' '.join(line_pred_parts)}\n\n"
        )
        vtt_entries.append(vtt_entry)

    return "".join(vtt_entries)

# Usage remains the same
def generate_subtitles(segment_index, output_path):
    vtt_content = create_line_by_line_vtt(
        segment_index,
        max_line_length=15,  # Adjust based on your preference
        line_duration=1500   # How long each line stays (ms)
    )

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(vtt_content)


In [72]:
for i in [4, 89]:
    output_file = f'{i}.vtt'
    generate_subtitles(segment_index, output_file)